# 03 — Microstructure (P2)

**What this notebook answers**

- Quoted spread distribution (bps) by venue.
- Top-of-book depth and depth profile.
- Order-book imbalance: does it predict short-horizon mid moves?
- Top-of-book churn (quote updates per second) — proxy for venue tick speed.

These set the **passive-quote regime** the MM is competing in. A 1-bps spread
on Binance vs 5-bps on HL means very different fill/edge math.

In [ ]:
from hlanalysis.analysis import duck, glob_for, load_df, set_mpl_defaults
import pandas as pd, numpy as np, matplotlib.pyplot as plt
set_mpl_defaults()

def bbo_df(venue, product_type, symbol):
    g = glob_for(venue=venue, product_type=product_type, event='bbo', symbol=symbol)
    df = load_df(f'''
        SELECT exchange_ts, bid_px, bid_sz, ask_px, ask_sz
        FROM read_parquet('{g}', hive_partitioning=true)
        WHERE bid_px>0 AND ask_px>0
        ORDER BY exchange_ts
    ''')
    if df.empty: return df
    df['t'] = pd.to_datetime(df.exchange_ts, unit='ns', utc=True)
    df['mid'] = 0.5*(df.bid_px+df.ask_px)
    df['spread_bps'] = 1e4*(df.ask_px-df.bid_px)/df.mid
    df['imbalance'] = (df.bid_sz - df.ask_sz)/(df.bid_sz + df.ask_sz)
    return df.set_index('t')

streams = {
    'HL perp BTC':       ('hyperliquid','perp','BTC'),
    'Binance perp BTC':  ('binance','perp','BTCUSDT'),
    'Binance spot BTC':  ('binance','spot','BTCUSDT'),
    'HL HIP-4 #30':      ('hyperliquid','prediction_binary','#30'),
    'HL HIP-4 #31':      ('hyperliquid','prediction_binary','#31'),
}
data = {k: bbo_df(*v) for k,v in streams.items()}
{k: len(v) for k,v in data.items()}

## 1. Quoted-spread distribution

In [ ]:
fig, ax = plt.subplots(figsize=(11,4))
for k, df in data.items():
    if df.empty: continue
    s = df.spread_bps.clip(upper=df.spread_bps.quantile(0.999))
    ax.hist(s, bins=120, histtype='step', label=f'{k}  median={s.median():.2f} bps', lw=1.4, density=True)
ax.set_xscale('log'); ax.set_xlabel('quoted spread (bps, log)'); ax.set_ylabel('density')
ax.set_title('Quoted spread distribution by venue'); ax.legend()
plt.tight_layout(); plt.show()

## 2. Quoted spread time series (1s resampled)

In [ ]:
fig, ax = plt.subplots(figsize=(11,4))
for k, df in data.items():
    if df.empty: continue
    ax.plot(df.spread_bps.resample('1s').median(), label=k, lw=0.8)
ax.set_yscale('log'); ax.set_ylabel('median quoted spread (bps)')
ax.set_title('Quoted spread over time'); ax.legend()
plt.tight_layout(); plt.show()

## 3. Top-of-book depth (bid_sz + ask_sz) over time

In [ ]:
fig, ax = plt.subplots(figsize=(11,4))
for k, df in data.items():
    if df.empty: continue
    tob = (df.bid_sz + df.ask_sz).resample('5s').median()
    ax.plot(tob, label=k, lw=0.8)
ax.set_yscale('log'); ax.set_ylabel('top-of-book size (base units)')
ax.set_title('Top-of-book depth'); ax.legend()
plt.tight_layout(); plt.show()

## 4. Depth profile from L2 snapshots

For each venue with `book_snapshot` available, plot cumulative bid/ask size as
a function of distance from mid. Use the most recent snapshot.

In [ ]:
def latest_book(venue, product_type, symbol):
    g = glob_for(venue=venue, product_type=product_type, event='book_snapshot', symbol=symbol)
    df = load_df(f'''
        SELECT * FROM read_parquet('{g}', hive_partitioning=true)
        ORDER BY exchange_ts DESC LIMIT 1
    ''')
    if df.empty: return None
    r = df.iloc[0]
    return r

books = {
    'HL perp BTC':      latest_book('hyperliquid','perp','BTC'),
    'Binance perp BTC': latest_book('binance','perp','BTCUSDT'),
    'Binance spot BTC': latest_book('binance','spot','BTCUSDT'),
}

fig, ax = plt.subplots(figsize=(11,4.5))
for label, r in books.items():
    if r is None: continue
    bid_px = np.asarray(r['bid_px'], dtype=float); bid_sz = np.asarray(r['bid_sz'], dtype=float)
    ask_px = np.asarray(r['ask_px'], dtype=float); ask_sz = np.asarray(r['ask_sz'], dtype=float)
    if len(bid_px)==0 or len(ask_px)==0: continue
    mid = 0.5*(bid_px[0]+ask_px[0])
    bid_dist = 1e4*(mid - bid_px)/mid
    ask_dist = 1e4*(ask_px - mid)/mid
    ax.plot(-bid_dist, np.cumsum(bid_sz), label=f'{label} bids')
    ax.plot( ask_dist, np.cumsum(ask_sz), label=f'{label} asks')
ax.set_xlabel('distance from mid (bps; negative=bid)'); ax.set_ylabel('cumulative size')
ax.set_xscale('symlog'); ax.set_yscale('log')
ax.set_title('L2 depth profile (latest snapshot)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Imbalance → next-mid-move

Sign-correlation of book imbalance with the **next** mid return at 1s/5s. If
this is meaningfully positive, simple skew is a free predictor for an MM.

In [ ]:
def imb_predict(df, horizons=(1,5,15)):
    if df.empty: return pd.DataFrame()
    s = df[['mid','imbalance']].copy().resample('1s').last().dropna()
    rows = []
    for h in horizons:
        ret = (s['mid'].shift(-h) - s['mid'])/s['mid']
        c = s['imbalance'].corr(ret)
        rows.append({'horizon_s': h, 'corr': c})
    return pd.DataFrame(rows)

out = []
for k, df in data.items():
    if df.empty: continue
    r = imb_predict(df)
    r.insert(0, 'stream', k)
    out.append(r)
pd.concat(out, ignore_index=True) if out else 'no data'

## 6. Top-of-book churn (BBO updates per second)

Higher = more aggressive market: harder to be passive without getting picked off.

In [ ]:
churn = {}
for k, df in data.items():
    if df.empty: continue
    churn[k] = df['mid'].resample('1s').count()
fig, ax = plt.subplots(figsize=(11,3.6))
for k, s in churn.items():
    ax.plot(s.rolling(30).median(), label=f'{k}  median={s.median():.1f}/s', lw=0.8)
ax.set_ylabel('BBO updates / s (30 s med)'); ax.set_title('TOB churn'); ax.legend()
plt.tight_layout(); plt.show()

## Strategy hypotheses

- **Supports:** H?: (fill in after reviewing the chart above)
- **Rules out:** H?: (fill in if evidence rules it out)
- **Next probe:** (what would refine this further)